# Project: MARBL Configuration - Plankton Time!

This project is about setting up a MARBL configuration and any nuances we might want to consider.

Download this project by either:

1. Running the CrocoDash CLI command: `crocodash template --machine derecho --notebook crocodash.projects.marbl`
2. Copying the file from your CrocoDash checkout: `demos/crocodash/projects/marbl.ipynb`

Grid Generation - MARBL is integrated into MOM6, so it's pretty straighforward to setup, there's not many nuances. Because there are so many tracers, we currently only support using forcing from old CESM runs, which do give data globally. 

So since MARBL adds cost, we will pick a small domain and run anywhere in the world. Let's rip through the steps!

In [ ]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.vgrid import VGrid
from CrocoDash.topo import Topo
from CrocoDash.case import Case


grid = Grid(
  resolution = 0.05, # in degrees
  xstart = 278.0, # min longitude in [0, 360]
  lenx = 5.0, # longitude extent in degrees
  ystart = 7.0, # min latitude in [-90, 90]
  leny = 5.0, # latitude extent in degrees
  name = "clear_marbl",
)

topo = Topo(
    grid = grid,
    min_depth = 9.5, # in meters
)

topo.set_from_dataset(    
    bathymetry_path = "<GEBCO_LOWRES>",
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation"
)

topo.depth.plot()
vgrid  = VGrid.hyperbolic(
    nk = 75, # number of vertical levels
    depth = topo.max_depth,
    ratio=20.0 # target ratio of top to bottom layer thicknesses
)


In [ ]:
# CESM case (experiment) name
casename = "lost_my_marbles"

# CESM source root (Update this path accordingly!!!)
cesmroot ="<CESM>"

# Place where all your input files go 
inputdir = Path("<inputdir>") / casename
    
# CESM case directory
caseroot = Path("<casedir>") / casename


case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = '<PROJECT>',
    override = True,
    machine = "derecho",
    compset = "CR1850MARBL_JRA" )


## MARBL Forcing

We use a global MARBL file to setup the initial condition (It is regridded as part of the model run), and generate OBCs in the same exact way as the MOM6 obc (so we're just adding >40 more tracers) We need to provide the path to a MARBL run or have it default to a run. What we're going to use here defaults to a FIESTY run (https://www.sciencedirect.com/science/article/abs/pii/S0079661124001204). We do not support having seperate physical vs MARBL forcings. All the forcings come from one product, in this case CESM POP Output, the old ocean model.

In [ ]:
case.configure_forcings(
    date_range=["2000-01-01 00:00:00", "2000-02-01 00:00:00"],
    product_name="CESM_POP_OUTPUT",
    function_name="get_cesm_single_variable_data",
    marbl_ic_filepath="/glade/campaign/collections/gdex/data/d651077/cesmdata/inputdata/ocn/mom/tx0.66v1/ecosys_jan_IC_omip_latlon_1x1_180W_c231221.nc",   
)


In [ ]:
case.process_forcings()


I'd mess with the NTASKS/ROOTPE's (give more ocean PEs) and build & submit!

Feel free to iterate with process forcings! You can run it on the command line with `crocodash process --caseroot YOURCASEROOT`!